# 静的診断 — 求解前にモデルから分かること(スケール・ブロック構造・対称性)

これまでのnotebook(1〜3)は「実際に解いて」得られる動的な観測量を扱った。
`minlpkit.collectors.static_diag` と `minlpkit.collectors.symmetry` は逆に、
**1度も分枝限定を走らせずにモデルの係数・接続構造だけから**数値健全性と分解適性を診断する。
3種類の静的診断を、それぞれ最も特徴が際立つモデルで見る。

1. **係数スケール**(`extract_coefficients`/`scale_summary`) — Big-M候補の検出。
   `samples/scheduling/unit_commitment.py`(既知のBig-M比840)
2. **ブロック構造**(`incidence`/`reorder_blocks`/`linking_constraints`) — 分解適性。
   `samples/others/scheduling_plant.py`(ジョブ横断の結合制約が既知)
3. **対称性**(`detect_symmetry`) — 入替可能な変数群。
   `samples/others/parallel_machines.py`(恒等並列機械、検証用に新規作成済みの対称モデル)

3つとも `numerical_scale` / `decomposable` / `symmetry_info` 診断ルールの一次情報になる。

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others", "samples/scheduling"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import unit_commitment as uc
import scheduling_plant as sp
import parallel_machines as pm
from minlpkit.collectors.static_diag import (
    extract_coefficients, scale_summary, incidence, reorder_blocks, linking_constraints,
    residual_scale)
from minlpkit.collectors.symmetry import detect_symmetry

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
SEQ_BLUE = [[0.0, C["surface"]], [1.0, C["s1"]]]
SRC_COLOR = {"制約係数": C["s1"], "RHS/LHS": C["s2"], "目的係数": "#e87ba4", "変数境界": "#eda100"}
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 係数スケール — Big-M候補の検出

`extract_coefficients` は線形制約の係数・RHS/LHS・目的係数・変数境界の絶対値を出所付きで
1本の `DataFrame` に集める。`scale_summary` は全体のmax/min比と、**中央値の100倍を超える
制約係数/変数境界**をBig-M候補として抽出する。ユニットコミットメント(`unit_commitment.py`)は
ランプ制約の `pmax=400` が既知のBig-M候補。

In [2]:
df_uc = extract_coefficients(uc.build_model())
s_uc = scale_summary(df_uc)
print(f"係数レンジ |min|={s_uc['min']:.3g} |max|={s_uc['max']:.3g} 比={s_uc['ratio']:.3g}")
print(f"Big-M候補({len(s_uc['bigm'])}件):",
     ", ".join(f"{b['name']}={b['magnitude']:.0f}" for b in s_uc["bigm"][:5]))

fig = go.Figure()
for src, color in SRC_COLOR.items():
    dd = df_uc[df_uc["source"] == src]
    if dd.empty:
        continue
    fig.add_trace(go.Box(y=dd["magnitude"], name=src, marker=dict(color=color), boxpoints="outliers"))
fig.update_layout(
    title=dict(text="unit_commitment: 係数の絶対値レンジ(出所別・対数)— 外れ点=Big-M候補",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), showlegend=False,
    yaxis=dict(title="|値|(対数)", type="log", gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"])),
    margin=dict(l=60, r=16, t=44, b=36), height=340)
show(fig)

係数レンジ |min|=1 |max|=840 比=840
Big-M候補(10件): c2=400, c12=400, c24=400, c16=400, c52=400


presolve前の比だけで「悪条件」と判断すると誤検出になる。SCIPのpresolveが自動で
Big-Mを締める場合があるため、`numerical_scale` ルールは **presolve後の残存スケール**
(`residual_scale`)で判断する。unit_commitmentのランプ制約は残存する(=presolveで
締まりきらない)ため実際に発火対象になる。

In [3]:
res = residual_scale(uc.build_model())
print(f"presolve前 比={s_uc['ratio']:.3g}  ->  presolve後(残存) 比={res['ratio']:.3g}")
report = mk.analyze(uc.build_model, name="uc-static", time_limit=10)
fired = [f for f in report.findings if f["id"] == "numerical_scale"]
print("numerical_scale 発火:", bool(fired))
if fired:
    print(" 根拠:", fired[0]["evidence"])

presolve前 比=840  ->  presolve後(残存) 比=2.62e+03


numerical_scale 発火: True
 根拠: presolve後の係数比=2.62e+03(presolve前840)、残存Big-M10件


## 2. ブロック構造 — 制約-変数の接続行列とRCM並べ替え

`incidence` は制約×変数の0/1接続行列を作る(`getConsVars` は非線形制約でも動く)。
これを二部グラフとみなし `reorder_blocks`(逆Cuthill-McKee)で並べ替えると、
分解可能な構造は**対角ブロック**として浮き上がる。`linking_constraints` は各制約が
何個の「変数グループ」(名前の末尾トークン、例 `J5`/`M2`)にまたがるかを数え、
多数のグループにまたがる制約 = 分解の境界となる**結合制約**を特定する。

In [4]:
model_plant = sp.build_model()
M, cn, vn = incidence(model_plant)
rp, cp, Mr = reorder_blocks(M)
fig = go.Figure(go.Heatmap(z=Mr, colorscale=SEQ_BLUE, showscale=False, xgap=0, ygap=0, hoverinfo="skip"))
fig.update_layout(
    title=dict(text=f"scheduling_plant: 制約-変数 接続行列(RCM並べ替え {M.shape[0]}×{M.shape[1]})"
                    " — 対角ブロック=分解可能",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="変数(並べ替え後)", showticklabels=False),
    yaxis=dict(title="制約(並べ替え後)", showticklabels=False, autorange="reversed"),
    margin=dict(l=60, r=16, t=44, b=36), height=420)
show(fig)

In [5]:
lk = linking_constraints(model_plant).head(12).iloc[::-1]
maxg = lk["n_groups"].max()
colors = ["#d03b3b" if g == maxg else C["s1"] for g in lk["n_groups"]]
fig = go.Figure(go.Bar(x=lk["n_groups"], y=lk["constraint"], orientation="h",
    marker=dict(color=colors), text=lk["n_groups"], textposition="outside",
    customdata=lk[["n_vars", "groups"]],
    hovertemplate="%{y}<br>またがるグループ数 %{x}<br>変数 %{customdata[0]}<extra></extra>"))
fig.update_layout(
    title=dict(text="結合制約 = またがる変数グループ数(赤=最多=分解の境界)",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="またがるグループ数", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"])),
    margin=dict(l=90, r=48, t=44, b=36), height=340)
show(fig)
print("最多結合制約:", lk.iloc[-1]["constraint"], "->", lk.iloc[-1]["groups"])

最多結合制約: load_M2 -> J1,J2,J3,J4,J5,J6,M2,cmax


`load_M1`/`load_M2`(マシンの負荷制約)が全ジョブグループにまたがる結合制約として
抽出される。これがベンダーズ/Dantzig-Wolfe分解の境界になる、という判断材料。
`decomposable` ルールは「最大結合が4グループ以上 かつ 重結合制約が3本以下」で発火する
(結合制約が少数に絞られているほど主問題を小さく保てる)。

## 3. 対称性 — 1-hop color refinementで入替可能な変数群を検出

`detect_symmetry` は各変数の(型・目的係数・境界・所属する全制約の形状と自身の係数)から
シグネチャを作り、同一シグネチャの変数群(サイズ≥2)を対称候補として返す。全制約が線形の
ときだけ健全(`sound=True`)に判定できる——非線形制約があると線形部分だけのシグネチャは
不完全で、定数差により対称性が崩れている可能性を否定できないため。
`parallel_machines.py` は恒等な機械(4台)へのジョブ割当で、意図的に強い対称性を持つ
検証用モデル(この収集器の健全性を確かめるために新規作成された)。

In [6]:
sym_df, sym_s = detect_symmetry(pm.build_model())
print(f"対称性: {'あり' if sym_s['has_symmetry'] else 'なし'}  健全判定: {sym_s['sound']}  "
     f"群数: {sym_s['n_groups']}  最大群サイズ: {sym_s['largest_group']}  "
     f"対称変数: {sym_s['n_symmetric_vars']}/{sym_s['n_linear_vars']}")

d = sym_df.copy()
d["label"] = d.apply(lambda r: f"群{r['signature_id']} ({r['size']}変数)", axis=1)
d = d.iloc[::-1]
fig = go.Figure(go.Bar(x=d["size"], y=d["label"], orientation="h",
    marker=dict(color="#4a3aa7"), text=d["size"], textposition="outside",
    customdata=d[["members"]], hovertemplate="%{y}<br>%{customdata[0]}<extra></extra>"))
fig.update_layout(
    title=dict(text="parallel_machines: 対称(入替可能)な変数群のサイズ",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="群内の変数数", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"]), dtick=1),
    margin=dict(l=140, r=48, t=44, b=36), height=max(220, 60 + 34 * len(d)))
show(fig)

対称性: あり  健全判定: True  群数: 6  最大群サイズ: 12  対称変数: 52/53


In [7]:
# 対照: 非線形制約を持つ plant では健全性フラグが False になる(偽陽性回避の仕組み)
_, sym_plant = detect_symmetry(model_plant)
print(f"plant(非線形あり): sound={sym_plant['sound']}  has_symmetry={sym_plant['has_symmetry']}"
     f"  caveat={sym_plant['caveat']}")

report = mk.analyze(pm.build_model, name="parallel-static", time_limit=10)
fired = [f for f in report.findings if f["id"] == "symmetry_info"]
print("symmetry_info 発火:", bool(fired), " (severity=good=対応不要の情報系)")

plant(非線形あり): sound=False  has_symmetry=None  caveat=非線形制約を含むため線形部分構造のみの判定(定数差で対称性が崩れる可能性=不確定)
symmetry_info 発火: True  (severity=good=対応不要の情報系)


`symmetry_info` は severity=`good`(=対応不要)の情報系ルール。SCIP内蔵の対称性処理
(既定ON)が典型インスタンスで自動的に対称性を解消するため、手動の辞書式順序制約は通常不要
——ここでも診断は「症状の検出」に留まり、対応の要否まで含めて正直に報告する。

## まとめ

- 3種の静的診断はいずれも **1度もsolveせずにモデルの記述だけから**得られる(対称性は
  presolve/solve不要、係数スケールも構築直後、ブロック構造も同様)。これは動的診断
  (notebook 1〜3)より軽量で、モデルを書いた直後にすぐ実行できる利点がある。
- 係数スケールはBig-M/数値不安定、ブロック構造は分解適性、対称性は探索木膨張のリスクを
  それぞれ指摘するが、**いずれもSCIPのpresolve/対称性処理/被約コスト固定が既に自動対応する
  範囲が広い**。だから診断はpresolve後の残存値(`residual_scale`)で判断し、対称性は
  `good`(参考情報)に留める——「本当に手を出す価値があるものだけ`warning`/`serious`に
  上げる」設計がこのリポジトリの一貫した方針。

関連: [プレイブック 0. 診断そのもの](../../playbook/00-diagnose.md) /
[プレイブック 8. 条件数・数値健全性](../../playbook/08-condition.md) /
[プレイブック 5. ベンダーズ分解](../../playbook/05-benders.md) /
API [`minlpkit.collectors`](../../api/pipeline.md)。